# Fetch New Agent PRs and Build Updated Dataset

This notebook:
1. Searches GitHub for **recent merged PRs authored by AI agents** (Claude Code, Copilot, Cursor, Devin, OpenAI Codex) using the same five-agent taxonomy as the original study.
2. Processes each PR through the exact same mining pipeline defined in `build.py` (Lizard metrics, Semgrep, documentation extraction, turnover).
3. Saves the result to `dataset/data/new_agent_dataset.csv` with the same 30-column schema as `final_dataset.csv`.
4. Merges `final_dataset.csv` + `new_agent_dataset.csv` → `dataset/data/final_dataset_new.csv`.

**Prerequisites:**
- A `.env` file in the project root with at least `GITHUB_TOKEN_1` (and optionally `GITHUB_TOKEN_2`, `GITHUB_TOKEN_3` for parallel processing).
- `lizard`, `semgrep`, `git` available on PATH.
- `pip install python-dotenv requests pandas tqdm textstat`

In [2]:
print('starting :)')

starting :)


In [3]:
# ── 0. Environment must be loaded BEFORE importing build.py (which checks tokens) ──
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Try several .env locations (notebook may be run from different CWDs)
for env_candidate in [
    Path('.') / '.env',
    Path('..') / '.env',
    Path('../..') / '.env',
]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        print(f'Loaded .env from {env_candidate.resolve()}')
        break

missing = [k for k in ['GITHUB_TOKEN_1'] if not os.environ.get(k)]
if missing:
    raise EnvironmentError(
        f"Missing required environment variables: {missing}\n"
        "Create a .env file in the project root with GITHUB_TOKEN_1=ghp_..."
    )
print('GitHub token(s) detected:', [k for k in ['GITHUB_TOKEN_1','GITHUB_TOKEN_2','GITHUB_TOKEN_3'] if os.environ.get(k)])

Loaded .env from G:\On the Naturalness of Agent-Generated Documentation\.env
GitHub token(s) detected: ['GITHUB_TOKEN_1']


In [4]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import re
import time
import json
import math
import csv
import shutil
import tempfile
import subprocess
import textwrap
import warnings
import numpy as np
import pandas as pd
import requests
import tqdm
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any

warnings.filterwarnings('ignore')

# ── Paths (works whether CWD is project root or this folder) ──────────────────
THIS_DIR = Path(os.path.abspath(''))
BUILD_DIR = THIS_DIR if (THIS_DIR / 'build.py').exists() else THIS_DIR / 'dataset' / 'buildDataset'
DATA_DIR  = BUILD_DIR.parent / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET_PATH    = DATA_DIR / 'final_dataset.csv'
NEW_AGENT_OUTPUT_PATH = DATA_DIR / 'new_agent_dataset.csv'
MERGED_OUTPUT_PATH    = DATA_DIR / 'final_dataset_new.csv'
PR_LIST_PATH          = DATA_DIR / 'new_agent_pr_list.parquet'

print('Build dir:', BUILD_DIR)
print('Data dir: ', DATA_DIR)
print('final_dataset.csv exists:', FINAL_DATASET_PATH.exists())

Build dir: g:\On the Naturalness of Agent-Generated Documentation\dataset\buildDataset
Data dir:  g:\On the Naturalness of Agent-Generated Documentation\dataset\data
final_dataset.csv exists: True


In [5]:
# ── 2. Import the mining pipeline from build.py ───────────────────────────────
# build.py must be importable — env tokens are already set above.
sys.path.insert(0, str(BUILD_DIR))

try:
    from build import (
        tokenize, calculate_entropy, doc_redundancy, doc_code_overlap,
        strip_comments, find_documentation_header, extract_documentation,
        parse_detailed_lizard, find_file_at_commit,
        get_time_based_shas, get_target_shas, getTurnover,
        AiDevMiner, REPO_BASE_DIR, SUPPORTED_EXTENSIONS, TOKENS,
    )
    print('Successfully imported pipeline from build.py')
    print(f'  REPO_BASE_DIR:        {REPO_BASE_DIR}')
    print(f'  SUPPORTED_EXTENSIONS: {len(SUPPORTED_EXTENSIONS)} types')
    print(f'  Active tokens:        {len(TOKENS)}')
except ImportError as e:
    raise ImportError(
        f'Could not import from build.py: {e}\n'
        f'Make sure {BUILD_DIR}/build.py exists and all its dependencies are installed.'
    )

GitHub token loaded successfully.
Successfully imported pipeline from build.py
  REPO_BASE_DIR:        cloned_repos
  SUPPORTED_EXTENSIONS: 18 types
  Active tokens:        1


## Step 1 — Fetch recent agent PRs from GitHub

We use the GitHub Search API with per-agent query signatures to find recently merged PRs authored by each AI agent. Adjust `TARGET_PRS_PER_AGENT` and `DATE_AFTER` to control the fetch size.

In [ ]:
# ── 3. Fetch configuration ────────────────────────────────────────────────────

TARGET_PRS_PER_AGENT = 200
MIN_STARS   = 25
MERGED_DATE = '2026-01-01'  # start of window (inclusive)
END_DATE    = datetime.today().strftime('%Y-%m-%d')  # today

# GitHub Search API — uses ONE primary token (rate limit: 30 requests/min)
SEARCH_TOKEN = os.environ.get('GITHUB_TOKEN_1') or TOKENS[0]
SEARCH_HEADERS = {
    'Authorization': f'token {SEARCH_TOKEN}',
    'Accept': 'application/vnd.github.v3+json',
}

# stars:>= is kept in the query but the date range is split into monthly windows
# so each sub-query stays under GitHub's 1000-result cap.
AGENT_QUERIES = [
    (
        'Claude_Code',
        f'is:pr is:merged stars:>={MIN_STARS} "Co-Authored-By: Claude"',
    ),
    (
        'Copilot',
        f'is:pr is:merged stars:>={MIN_STARS} head:copilot/',
    ),
    (
        'Cursor',
        f'is:pr is:merged stars:>={MIN_STARS} head:cursor/',
    ),
    (
        'Devin',
        f'is:pr is:merged stars:>={MIN_STARS} author:devin-ai-integration[bot]',
    ),
    (
        'OpenAI_Codex',
        f'is:pr is:merged stars:>={MIN_STARS} head:codex/',
    ),
]

print('Agent search configuration:')
for agent, q in AGENT_QUERIES:
    print(f'  {agent:<16} -> {q}')
print(f'\nDate window: {MERGED_DATE} → {END_DATE}')
print(f'Target: {TARGET_PRS_PER_AGENT} per agent  |  Min stars: {MIN_STARS}')

Agent search configuration:
  Claude_Code      -> is:pr stars:>=25 is:merged "Co-Authored-By: Claude"
  Copilot          -> is:pr is:merged stars:>=25 head:copilot/
  Cursor           -> is:pr is:merged stars:>=25 head:cursor/
  Devin            -> is:pr is:merged stars:>=25 author:devin-ai-integration[bot]
  OpenAI_Codex     -> is:pr is:merged stars:>=25 head:codex/

Date window: 2026-01-01 → 2026-05-09
Target: 200 per agent  |  Min stars: 25


In [30]:
# ── 4. GitHub PR fetcher (date-windowed) ──────────────────────────────────────
# Splits the date range into monthly windows and queries each separately.
# This bypasses the 1000-result cap per query while keeping stars:>= in the query,
# which would otherwise suppress results when combined with in:body text search.

def month_windows(start_date: str, end_date: str):
    """Yield (window_start, window_end) strings in YYYY-MM-DD format, month by month."""
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end   = datetime.strptime(end_date,   '%Y-%m-%d')
    cur   = start
    while cur <= end:
        nxt = (cur.replace(day=1) + timedelta(days=32)).replace(day=1)
        yield cur.strftime('%Y-%m-%d'), min(nxt - timedelta(days=1), end).strftime('%Y-%m-%d')
        cur = nxt


def fetch_agent_prs(agent_label: str, base_query: str, target: int,
                    start_date: str, end_date: str) -> List[dict]:
    """
    Fetch up to `target` merged PRs by querying one month at a time.
    Each window uses the full base_query (including stars:>=) plus a merged: range.
    """
    prs      = []
    seen_ids = set()

    for win_start, win_end in month_windows(start_date, end_date):
        if len(prs) >= target:
            break

        query    = f'{base_query} merged:{win_start}..{win_end}'
        page     = 1
        per_page = 100

        while len(prs) < target:
            params = {'q': query, 'per_page': per_page, 'page': page,
                      'sort': 'created', 'order': 'desc'}
            try:
                r = requests.get('https://api.github.com/search/issues',
                                 headers=SEARCH_HEADERS, params=params, timeout=20)
            except requests.RequestException as e:
                print(f'  Request error ({win_start}..{win_end} p{page}): {e}')
                break

            if r.status_code in (403, 429):
                wait = max(int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
                           - int(time.time()), 0) + 2
                print(f'  Rate limited. Waiting {wait}s...')
                time.sleep(wait)
                continue

            if r.status_code != 200:
                print(f'  API error {r.status_code}: {r.text[:200]}')
                break

            data  = r.json()
            items = data.get('items', [])
            if not items:
                break

            for item in items:
                if len(prs) >= target:
                    break
                item_id = item.get('id')
                if item_id in seen_ids:
                    continue
                seen_ids.add(item_id)
                merged_at = item.get('pull_request', {}).get('merged_at')
                if not merged_at:
                    continue
                prs.append({
                    'id':         item_id,
                    'number':     item.get('number'),
                    'title':      item.get('title'),
                    'body':       item.get('body'),
                    'agent':      agent_label,
                    'user_id':    item.get('user', {}).get('login'),
                    'user':       item.get('user', {}).get('login'),
                    'state':      item.get('state'),
                    'created_at': item.get('created_at'),
                    'closed_at':  item.get('closed_at'),
                    'merged_at':  merged_at,
                    'repo_id':    None,
                    'repo_url':   item.get('repository_url', ''),
                    'html_url':   item.get('html_url'),
                })

            time.sleep(2.2)
            page += 1
            total_available = data.get('total_count', 0)
            if page * per_page > min(1000, total_available):
                break

    return prs

In [33]:
# ── 5. Run the fetch for all agents ───────────────────────────────────────────
all_new_prs = []

for agent_label, query in AGENT_QUERIES:
    print(f'\nFetching {agent_label} PRs...')
    prs = fetch_agent_prs(agent_label, query, TARGET_PRS_PER_AGENT, MERGED_DATE, END_DATE)
    print(f'  -> {len(prs)} PRs collected')
    if len(prs) < TARGET_PRS_PER_AGENT:
        print(f'  WARNING: only {len(prs)}/{TARGET_PRS_PER_AGENT} found — widen MERGED_DATE or lower MIN_STARS')
    all_new_prs.extend(prs)

df_prs = pd.DataFrame(all_new_prs)
print(f'\nTotal PRs fetched: {len(df_prs)}')
print(df_prs['agent'].value_counts().to_string())


Fetching Claude_Code PRs...
  -> 200 PRs collected

Fetching Copilot PRs...


KeyboardInterrupt: 

In [8]:
# ── 5b. Post-fetch star filter + trim to TARGET_PRS_PER_AGENT ─────────────────
# We filter here (not in the query) because stars:>= changes GitHub's search
# indexing even at >=0, reducing available results for some agents.

def get_repo_stars(repo_api_url: str) -> int:
    try:
        r = requests.get(repo_api_url, headers=SEARCH_HEADERS, timeout=10)
        if r.status_code == 200:
            return r.json().get('stargazers_count', 0)
    except Exception:
        pass
    return 0

repo_star_cache = {}
unique_repos = df_prs['repo_url'].unique()
print(f'Fetching star counts for {len(unique_repos)} unique repos...')
for repo_url in tqdm.tqdm(unique_repos):
    repo_star_cache[repo_url] = get_repo_stars(repo_url)
    time.sleep(0.5)

df_prs['repo_stars'] = df_prs['repo_url'].map(repo_star_cache)

before = len(df_prs)
df_prs = df_prs[df_prs['repo_stars'] >= MIN_STARS].reset_index(drop=True)
print(f'\nDropped {before - len(df_prs)} PRs from repos with < {MIN_STARS} stars')

# Trim each agent to TARGET_PRS_PER_AGENT (preserve recency order from the fetch)
df_prs = (
    df_prs.groupby('agent', group_keys=False)
    .apply(lambda g: g.head(TARGET_PRS_PER_AGENT))
    .reset_index(drop=True)
)

print(f'After trim: {len(df_prs)} PRs')
counts = df_prs['agent'].value_counts()
print(counts.to_string())

short = counts[counts < TARGET_PRS_PER_AGENT]
if not short.empty:
    print(f'\nWARNING: the following agents have fewer than {TARGET_PRS_PER_AGENT} qualifying PRs:')
    print(short.to_string())
    print('Consider widening MERGED_DATE or lowering MIN_STARS.')

Fetching star counts for 2159 unique repos...


100%|██████████| 2159/2159 [32:40<00:00,  1.10it/s]


Dropped 4648 PRs from repos with < 10 stars
After trim: 324 PRs
agent
Claude_Code     136
Cursor           66
Copilot          53
OpenAI_Codex     52
Devin            17

agent
Claude_Code     136
Cursor           66
Copilot          53
OpenAI_Codex     52
Devin            17
Consider widening MERGED_DATE or lowering MIN_STARS.


In [9]:
# ── 6. Deduplicate against final_dataset.csv to avoid overlap ─────────────────
if FINAL_DATASET_PATH.exists():
    df_existing = pd.read_csv(FINAL_DATASET_PATH, usecols=['repo', 'pull_request'])
    # final_dataset repo column: https://api.github.com/repos/...
    # df_prs repo_url column:    https://api.github.com/repos/...
    existing_keys = set(
        zip(df_existing['repo'].astype(str), df_existing['pull_request'].astype(str))
    )

    before = len(df_prs)
    df_prs = df_prs[
        ~df_prs.apply(
            lambda row: (str(row['repo_url']), str(row['number'])) in existing_keys,
            axis=1
        )
    ].reset_index(drop=True)

    print(f'Removed {before - len(df_prs)} PRs already in final_dataset.csv')
else:
    print('final_dataset.csv not found — processing all fetched PRs')

print(f'PRs to process: {len(df_prs)}')

# Save PR list for reproducibility
df_prs.to_parquet(PR_LIST_PATH, index=False)
print(f'PR list saved to {PR_LIST_PATH}')

Removed 0 PRs already in final_dataset.csv
PRs to process: 324
PR list saved to g:\On the Naturalness of Agent-Generated Documentation\dataset\data\new_agent_pr_list.parquet


## Step 2 — Mine each PR through the build pipeline

This uses the same `AiDevMiner.process_pr()` from `build.py`. It clones each repository, runs Lizard for code metrics, Semgrep for quality checks, extracts documentation, and computes turnover at commit offsets C5/C10/C20 and months M1/M3.

**Expected runtime:** ~2–5 minutes per PR (dominated by git clone + Semgrep). With `TARGET_PRS_PER_AGENT=200` and 5 agents this can take several hours — reduce `TARGET_PRS_PER_AGENT` to `10` for a quick smoke test.

In [10]:
# ── 7. Run the mining pipeline ─────────────────────────────────────────────────
# Column order exactly matches final_dataset.csv
FINAL_COLUMNS = [
    'repo', 'pull_request', 'label', 'file_path', 'function_name',
    'function_start_line', 'function_end_line', 'function',
    'loc', 'sloc', 'cyclomatic_complexity', 'num_parameters',
    'doc_lines', 'doc_text', 'doc_entropy', 'total_entropy',
    'doc_readability', 'semgrep_findings', 'semgrep_findings_count',
    'doc_code_overlap', 'doc_redundancy',
    'pr_date_merged', 'pr_date_created', 'pr_date_closed',
    'turnover_c5', 'turnover_c10', 'turnover_c20',
    'turnover_m1', 'turnover_m3',
    'group',  # added post-process: always 'agent' for this dataset
]

miner      = AiDevMiner()
MAX_WORKERS = max(1, len(TOKENS))
final_stats = Counter()

with open(NEW_AGENT_OUTPUT_PATH, 'w', encoding='utf-8', newline='') as f:
    writer = None

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_pr = {
            executor.submit(miner.process_pr, row, TOKENS[i % len(TOKENS)]): row.get('number', '?')
            for i, (_, row) in enumerate(df_prs.iterrows())
        }

        for future in tqdm.tqdm(
            as_completed(future_to_pr),
            total=len(future_to_pr),
            desc='Mining agent PRs'
        ):
            try:
                result_pkg   = future.result()
                results_list = result_pkg['data']
                final_stats.update(result_pkg['stats'])

                for row_result in results_list:
                    row_result['group'] = 'agent'  # always 'agent' for this dataset

                    if writer is None:
                        writer = csv.DictWriter(f, fieldnames=FINAL_COLUMNS)
                        writer.writeheader()

                    writer.writerow({k: row_result.get(k) for k in FINAL_COLUMNS})

            except Exception as e:
                print(f'\n[ERROR] PR {future_to_pr[future]}: {e}')

print('\nMining complete.')
for k, v in sorted(final_stats.items()):
    print(f'  {k}: {v}')

stats_path = DATA_DIR / 'mining_stats_new_agent.json'
with open(stats_path, 'w') as sj:
    json.dump(dict(final_stats), sj, indent=4)
print(f'Stats saved to {stats_path}')

Mining agent PRs: 100%|██████████| 324/324 [2:50:02<00:00, 31.49s/it]   



Mining complete.
  fail_pr_fetch: 1
  files_processed: 170
  files_unsupported: 1402
  lizard_timeouts: 0
  skipped_clone_fail: 17
  skipped_too_many_files: 4
  total_files_seen: 2612
Stats saved to g:\On the Naturalness of Agent-Generated Documentation\dataset\data\mining_stats_new_agent.json


In [11]:
# ── 8. Inspect new_agent_dataset.csv ──────────────────────────────────────────
df_new = pd.read_csv(NEW_AGENT_OUTPUT_PATH)

print(f'new_agent_dataset.csv shape: {df_new.shape}')
print(f'Columns match final_dataset: {list(df_new.columns) == FINAL_COLUMNS}')
print()
print('Functions per agent label:')
print(df_new['label'].value_counts().to_string())
print()
print('Documentation rate (doc_lines > 0):')
for lbl in df_new['label'].unique():
    sub  = df_new[df_new['label'] == lbl]
    rate = (pd.to_numeric(sub['doc_lines'], errors='coerce') > 0).mean()
    print(f'  {lbl}: {rate:.1%}  (n={len(sub)})')
print()
display(df_new[['label','repo','pull_request','function_name','doc_lines','sloc','cyclomatic_complexity']].head())

new_agent_dataset.csv shape: (531, 30)
Columns match final_dataset: True

Functions per agent label:
label
OpenAI_Codex    197
Copilot         128
Claude_Code     119
Devin            81
Cursor            6

Documentation rate (doc_lines > 0):
  Claude_Code: 42.0%  (n=119)
  Copilot: 45.3%  (n=128)
  Cursor: 50.0%  (n=6)
  Devin: 19.8%  (n=81)
  OpenAI_Codex: 20.3%  (n=197)



,label,repo,pull_request,function_name,doc_lines,sloc,cyclomatic_complexity
0,Claude_Code,https://api.github.com/repos/hw-native-sys/sim...,727,generate_args,0,7,1
1,Claude_Code,https://api.github.com/repos/hw-native-sys/sim...,727,compute_golden,0,2,1
2,Claude_Code,https://api.github.com/repos/CTalkobt/M65Compiler,38,runCC45,4,15,2
3,Claude_Code,https://api.github.com/repos/CTalkobt/M65Compiler,38,test_fam_in_union,1,13,1
4,Claude_Code,https://api.github.com/repos/CTalkobt/M65Compiler,38,test_fam_not_last,1,13,1


## Step 3 — Merge with final_dataset.csv

In [6]:
# ── 9. Merge new_agent_dataset.csv + final_dataset.csv → final_dataset_new.csv ──

if not FINAL_DATASET_PATH.exists():
    raise FileNotFoundError(f'{FINAL_DATASET_PATH} not found. Cannot merge.')

df_original = pd.read_csv(FINAL_DATASET_PATH)
df_new      = pd.read_csv(NEW_AGENT_OUTPUT_PATH)

print(f'final_dataset.csv:     {df_original.shape}')
print(f'new_agent_dataset.csv: {df_new.shape}')
print()
print('Group breakdown in original dataset:')
print(df_original['group'].value_counts().to_string())

# Align columns before concat (handles any ordering differences)
df_new_aligned = df_new.reindex(columns=df_original.columns)

df_merged = pd.concat([df_original, df_new_aligned], ignore_index=True)

# Deduplicate on the natural key — file_path must be included so that
# anonymous functions in different files of the same PR are not collapsed.
before    = len(df_merged)
df_merged = df_merged.drop_duplicates(
    subset=['repo', 'pull_request', 'file_path', 'function_name', 'function_start_line'],
    keep='first'
).reset_index(drop=True)

if before - len(df_merged):
    print(f'Dropped {before - len(df_merged)} duplicate rows.')

print()
print(f'Merged dataset shape: {df_merged.shape}')
print('Group breakdown in merged dataset:')
print(df_merged['group'].value_counts().to_string())
print('Label breakdown in merged dataset:')
print(df_merged['label'].value_counts().to_string())

df_merged.to_csv(MERGED_OUTPUT_PATH, index=False)
print(f'\nSaved to: {MERGED_OUTPUT_PATH}')


final_dataset.csv:     (13467, 30)
new_agent_dataset.csv: (531, 30)

Group breakdown in original dataset:
group
human    6918
agent    6549
Dropped 189 duplicate rows.

Merged dataset shape: (13809, 30)
Group breakdown in merged dataset:
group
agent    6975
human    6834
Label breakdown in merged dataset:
label
Claude_Code     2458
Copilot         1634
Cursor          1342
Devin            986
OpenAI_Codex     555

Saved to: g:\On the Naturalness of Agent-Generated Documentation\dataset\data\final_dataset_new.csv
